In [ ]:
# !pip install requests pandas sentence-transformers hdbscan google-generativeai jupyter

In [ ]:
# !pip install streamlit requests sentence-transformers hdbscan pandas numpy google-genai

In [ ]:
# %pip install google-genai

In [ ]:
from sentence_transformers import SentenceTransformer
import hdbscan
import requests
import pandas as pd
import os
from google import genai

# --- Credentials ---
BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [ ]:
def fetch_bluesky_posts(query, target_count=1000):
    print(f"Authenticating as {BSKY_HANDLE}...")
    
    # 1. Create a session to get the auth token
    session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    session_data = {"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD}
    session_resp = requests.post(session_url, json=session_data).json()
    
    if "accessJwt" not in session_resp:
        raise Exception(f"Failed to authenticate: {session_resp}")
        
    auth_token = session_resp["accessJwt"]
    headers = {"Authorization": f"Bearer {auth_token}"}
    
    # 2. Search for posts iteratively
    search_url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    
    posts_data = []
    cursor = None
    
    print(f"Fetching {target_count} posts for '{query}'...")
    while len(posts_data) < target_count:
        params = {"q": query, "limit": 100} # 100 is the max per request
        if cursor:
            params["cursor"] = cursor
            
        resp = requests.get(search_url, headers=headers, params=params).json()
        new_posts = resp.get("posts", [])
        
        if not new_posts:
            break # No more posts available
            
        for post in new_posts:
            # We extract the text, the timestamp, and the author
            posts_data.append({
                "text": post["record"]["text"],
                "created_at": post["record"]["createdAt"],
                "author": post["author"]["handle"]
            })
            
        cursor = resp.get("cursor")
        if not cursor:
            break
            
    # Keep only the target amount and convert to a DataFrame
    df = pd.DataFrame(posts_data[:target_count])
    print(f"Successfully fetched {len(df)} posts.")
    return df

# Test
df_posts = fetch_bluesky_posts("skincare", target_count=1000)
df_posts.head()

In [ ]:
def cluster_social_posts(df):
    print("Loading Sentence Transformer model...")

    model = SentenceTransformer('all-MiniLM-L6-v2') 
    
    embeddings = model.encode(df['text'].tolist())
    
    print("Running HDBSCAN clustering...")
    # min_cluster_size dictates how many posts are needed to form a "trend"
    clusterer = hdbscan.HDBSCAN(min_cluster_size=15, metric='euclidean')
    df['cluster_id'] = clusterer.fit_predict(embeddings)
    
    # -1 means "noise" (unclustered). Let's filter those out.
    clustered_df = df[df['cluster_id'] != -1]
    
    return clustered_df

# Test
df_clustered = cluster_social_posts(df_posts)
print(df_clustered['cluster_id'].value_counts())

In [ ]:
def label_clusters_with_gemini(df_clustered):
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    cluster_labels = {}
    unique_clusters = df_clustered['cluster_id'].unique()
    
    for cluster_id in unique_clusters:
        # Get 5 random posts from this cluster to show the LLM
        sample_posts = df_clustered[df_clustered['cluster_id'] == cluster_id]['text'].head(5).tolist()
        posts_text = "\n- ".join(sample_posts)
        
        prompt = f"""
        You are a consumer trend analyst. Look at the following social media posts that have been clustered together:
        - {posts_text}
        
        What is the specific, underlying trend or topic they are discussing? 
        Provide a catchy, 2-to-4 word label for this cluster. Do not include quotes or extra text.
        """
        
        # Using Gemini 2.0 Flash for blazing fast text generation
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt
        )
        
        cluster_labels[cluster_id] = response.text.strip()
        print(f"Cluster {cluster_id} labeled as: {cluster_labels[cluster_id]}")
        
    # Map the labels back to the dataframe
    df_clustered['trend_label'] = df_clustered['cluster_id'].map(cluster_labels)
    return df_clustered


In [ ]:

# Test it
df_labeled = label_clusters_with_gemini(df_clustered)